> **⚠️ OUTDATED (2026-07-16) — xem trước khi chạy lại**
>
> Notebook này đọc `data/signals/meta_ranking_watchlist.parquet`/`meta_ranking.parquet` như một xếp hạng
> có ý nghĩa xuyên suốt lịch sử. `WATCHLIST_VALIDATION_REPORT.md` (GĐ4) phát hiện `meta_ranking.parquet`
> thực chất là **snapshot trạng thái hiện tại** (mỗi mã 1 dòng, không phải chuỗi thời gian) — không dùng
> được để suy diễn/kiểm định qua các mốc lịch sử như notebook này giả định.
>
> Xem thêm: `WATCHLIST_VALIDATION_REPORT.md` mục 2 phát hiện #4.
>
> **Không sửa nội dung notebook trong round này** — chỉ cảnh báo trước khi dùng lại kết quả.

# 19 — Quantitative System Validation & Stress Test

Kiểm định hệ thống lượng hóa, rà soát data leakage, look-ahead bias và stress testing.

In [1]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path

def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")

project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [2]:
# Cell: Import Validation Modules
import pandas as pd
import numpy as np

from src.data.loaders import read_parquet
from src.portfolio import (
    calculate_returns_matrix,
    estimate_expected_returns,
    calculate_portfolio_volatility
)
from src.validation import (
    check_lookahead_bias,
    check_data_leakage,
    check_survivorship_bias,
    run_parameter_sensitivity,
    run_cost_sensitivity,
    run_risk_limit_sensitivity,
    analyze_performance_by_regime,
    run_market_drawdown_stress,
    run_correlation_shock,
    run_volatility_shock,
    generate_watchlist_explanation,
    calculate_time_under_water,
    calculate_cagr,
    calculate_sharpe,
    calculate_max_drawdown,
)

In [3]:
# Cell: Load Inputs
signals = read_parquet("data/signals/setup_signals.parquet")
clean_daily = read_parquet("data/processed/clean_daily_ohlcv.parquet")
weights_df = read_parquet("data/signals/portfolio_construction_weights.parquet")
trade_cards = read_parquet("data/signals/breakout_20d_trade_cards.parquet")
overlay_df = read_parquet("data/signals/risk_overlay_weights.parquet")

signals["date"] = pd.to_datetime(signals["date"])
clean_daily["date"] = pd.to_datetime(clean_daily["date"])

print(f"Signals rows: {len(signals)}")
print(f"Weights symbols: {len(weights_df)}")

Signals rows: 186827
Weights symbols: 10


## 1. Out-of-Sample Testing

In [4]:
# Cell: Out-of-Sample Performance Splits
symbols = weights_df["symbol"].tolist()
w_series = weights_df.set_index("symbol")["max_sharpe"]

return_matrix = calculate_returns_matrix(signals[signals["symbol"].isin(symbols)])
return_matrix = return_matrix.dropna(how="all").fillna(0.0)

# Portfolio returns series
port_returns = return_matrix.reindex(columns=w_series.index).mul(w_series, axis=1).sum(axis=1)
port_returns.name = "portfolio_return"

# Define time segments
segments = {
    "Train (2020-2022)": port_returns.loc[:"2022-12-31"],
    "Validation (2023-2024)": port_returns.loc["2023-01-01":"2024-12-31"],
    "Out-of-Sample (2025-2026)": port_returns.loc["2025-01-01":]
}

segment_results = []
for name, ret in segments.items():
    if ret.empty:
        continue
    nav = (1 + ret).cumprod()
    cagr = calculate_cagr(nav)
    sharpe = calculate_sharpe(ret)
    mdd = calculate_max_drawdown(nav)
    tuw = calculate_time_under_water(nav)
    
    segment_results.append({
        "Segment": name,
        "Days": len(ret),
        "CAGR (Annualized)": f"{cagr:.2%}",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "Max Drawdown": f"{mdd:.2%}",
        "Time Under Water": f"{tuw:.2%}"
    })

pd.DataFrame(segment_results)

,Segment,Days,CAGR (Annualized),Sharpe Ratio,Max Drawdown,Time Under Water
0,Train (2020-2022),750,3.44%,0.09,-8.38%,91.87%
1,Validation (2023-2024),499,38.18%,1.95,-12.12%,79.36%
2,Out-of-Sample (2025-2026),366,45.24%,2.09,-13.13%,79.78%


## 2. Bias & Leakage Checks

In [ ]:
# Cell: Run Leakage Verification
lookahead_check = check_lookahead_bias(signals)
print(f"Look-ahead Bias check passed: {lookahead_check['passed']} ({lookahead_check['details']})")

train_data = signals[signals["date"] <= "2023-12-31"]
test_data = signals[signals["date"] >= "2024-01-01"]
leakage_check = check_data_leakage(train_data, test_data)
print(f"Data Leakage check passed: {leakage_check['passed']} ({leakage_check['details']})")

# Survivorship check: requires a REAL delisted-symbol list (audit D3/D4).
# No mock/fabricated list -- if the data does not exist yet, say so explicitly
# instead of faking a pass against 3 hardcoded names.
delisted_path = project_root / "config" / "delisted_symbols.csv"
if delisted_path.exists():
    delisted_symbols = pd.read_csv(delisted_path)["symbol"].astype(str).tolist()
    survivorship_check = check_survivorship_bias(signals["symbol"].unique(), delisted_symbols)
    print(f"Survivorship Bias check details: {survivorship_check['details']}")
else:
    print("SKIPPED: no real delisted-symbol data found at config/delisted_symbols.csv "
          "(see AUDIT_REPORT.md D3) -- survivorship bias is not being measured.")


## 3. Robustness & Sensitivity

In [6]:
# Cell: Sweeps and Sensitivities
from src.backtest.breakout_research import run_breakout_research
res = run_breakout_research(clean_daily)
entries = res["non_overlap_entries"]
prepared = res["prepared_features"]

print("--- Parameter Sensitivity Summary ---")
sensitivity_df = run_parameter_sensitivity(entries, prepared, holding_periods=[10, 20], stop_losses=[-0.05, -0.10])
display(sensitivity_df)

print("\n--- Cost Sensitivity Summary ---")
cost_df = run_cost_sensitivity(entries, prepared, costs=[0.0, 0.004, 0.008])
display(cost_df)

print("\n--- Risk Limit Sensitivity Summary ---")
expected_returns = estimate_expected_returns(return_matrix, annualized=True)
cov_matrix_annual = return_matrix.cov() * 252
risk_df = run_risk_limit_sensitivity(expected_returns, cov_matrix_annual, risk_limits=[0.08, 0.10, 0.12])
display(risk_df)

--- Parameter Sensitivity Summary ---


,holding_period,stop_loss,trades,win_rate,profit_factor,avg_return
0,10,-0.05,3754,0.368141,1.142585,0.004071
1,10,-0.10,3754,0.413958,1.095826,0.003149
2,20,-0.05,3754,0.315131,1.164812,0.005734
3,20,-0.10,3754,0.382525,1.035611,0.001609



--- Cost Sensitivity Summary ---


,cost_pct,trades,win_rate,profit_factor,avg_return
0,0.000,3754,0.393181,1.131238,0.005609
1,0.004,3754,0.382525,1.035611,0.001609
2,0.008,3754,0.367075,0.949858,-0.002391



--- Risk Limit Sensitivity Summary ---


,risk_limit,expected_return,realized_volatility,sharpe_ratio
0,0.08,0.148011,2.628749,0.044892
1,0.10,0.148011,2.628749,0.044892
2,0.12,0.148011,2.628749,0.044892


## 4. Regime Analysis

In [7]:
# Cell: Performance by Regime
best_fixed_trades = res["best_fixed_trades"]
regime_perf = analyze_performance_by_regime(best_fixed_trades, regime_col="volatility_regime")
print("--- Strategy Performance by Volatility Regime ---")
display(regime_perf)

--- Strategy Performance by Volatility Regime ---


,volatility_regime,trades,win_rate,avg_return,median_return,best_return,worst_return,gross_profit,gross_loss,profit_factor,stop_rate
0,Low Vol,296,0.327703,0.018229,-0.054000,1.211806,-0.185829,16.162011,10.766311,1.501165,0.496622
1,Panic Vol,134,0.410448,0.010973,-0.031867,0.449554,-0.149161,5.261824,3.791469,1.387806,0.447761
2,Stress Vol,137,0.350365,0.005912,-0.054000,0.544387,-0.174483,5.417480,4.607489,1.175799,0.489051
3,Unknown,2640,0.299621,0.004700,-0.054000,3.262667,-0.309556,105.028745,92.619554,1.133980,0.583712
4,High Vol,221,0.352941,0.003987,-0.054000,0.549539,-0.287140,8.368912,7.487698,1.117688,0.493213
5,Normal Vol,326,0.349693,0.001713,-0.054000,0.730042,-0.169605,11.889125,11.330665,1.049288,0.447853


## 5. Stress Testing

In [8]:
# Cell: Volatility and Correlation Shocks
w_equal = pd.Series(1.0 / len(symbols), index=symbols)
w_opt = weights_df.set_index("symbol")["max_ret_under_risk"]

normal_vol_opt = calculate_portfolio_volatility(w_opt, cov_matrix_annual, annualized=True)
normal_vol_eq = calculate_portfolio_volatility(w_equal, cov_matrix_annual, annualized=True)

shocked_vol_opt = run_volatility_shock(w_opt, cov_matrix_annual, multiplier=1.5)
shocked_vol_eq = run_volatility_shock(w_equal, cov_matrix_annual, multiplier=1.5)

shocked_corr_opt = run_correlation_shock(w_opt, cov_matrix_annual, multiplier=1.5)
shocked_corr_eq = run_correlation_shock(w_equal, cov_matrix_annual, multiplier=1.5)

stress_losses = run_market_drawdown_stress(w_opt, return_matrix, market_drop=-0.10)

stress_table = pd.DataFrame([
    {"Portfolio": "Equal-Weight", "Normal Vol": f"{normal_vol_eq:.2%}", "Volatility Shock (1.5x)": f"{shocked_vol_eq:.2%}", "Correlation Shock (1.5x)": f"{shocked_corr_eq:.2%}"},
    {"Portfolio": "Optimized (Obj 1)", "Normal Vol": f"{normal_vol_opt:.2%}", "Volatility Shock (1.5x)": f"{shocked_vol_opt:.2%}", "Correlation Shock (1.5x)": f"{shocked_corr_opt:.2%}"}
])
print("--- Volatility & Correlation Stress Test ---")
display(stress_table)
print(f"Expected portfolio loss under -10% market crash: {stress_losses['expected_portfolio_loss']:.2%}")

--- Volatility & Correlation Stress Test ---


,Portfolio,Normal Vol,Volatility Shock (1.5x),Correlation Shock (1.5x)
0,Equal-Weight,262.87%,394.31%,300.28%
1,Optimized (Obj 1),162.67%,244.01%,175.11%


Expected portfolio loss under -10% market crash: -4.65%


## 6. Explainability

In [9]:
# Cell: Explain Watchlist Decisions
watchlist_ranked = read_parquet("data/signals/meta_ranking_watchlist.parquet")
explanations = generate_watchlist_explanation(watchlist_ranked)

print("--- Watchlist Selected Stock Explanations ---")
for idx, exp in enumerate(explanations[:5]):
    print(f"\nStock #{idx+1}:")
    print(exp)

--- Watchlist Selected Stock Explanations ---

Stock #1:
Symbol: VHM (Meta Score: 63.74)
  - Xu hướng (Trend Score): 50.00/100.0
  - Động lượng (Momentum Score): 50.00/100.0
  - Sức mạnh giá so với VNINDEX (RS Score): 77.12/100.0
  - Trạng thái biến động: Normal Vol
  - Lý do hệ thống chọn: N/A
  - Cảnh báo đi kèm: N/A

Stock #2:
Symbol: STB (Meta Score: 62.92)
  - Xu hướng (Trend Score): 50.00/100.0
  - Động lượng (Momentum Score): 50.00/100.0
  - Sức mạnh giá so với VNINDEX (RS Score): 88.92/100.0
  - Trạng thái biến động: Normal Vol
  - Lý do hệ thống chọn: N/A
  - Cảnh báo đi kèm: N/A

Stock #3:
Symbol: POW (Meta Score: 60.55)
  - Xu hướng (Trend Score): 50.00/100.0
  - Động lượng (Momentum Score): 50.00/100.0
  - Sức mạnh giá so với VNINDEX (RS Score): 85.32/100.0
  - Trạng thái biến động: Normal Vol
  - Lý do hệ thống chọn: N/A
  - Cảnh báo đi kèm: N/A

Stock #4:
Symbol: VIC (Meta Score: 60.08)
  - Xu hướng (Trend Score): 50.00/100.0
  - Động lượng (Momentum Score): 50.00/100.0
 